# Reģistru reizināšana - RGQFTMultiplier

RGQFTMultiplier ir gatavs Qiskit reizinātāšanas algoritms, kas paņem divus $n$-kubit reģistrus $a$ un $b$, un rezultātu ieraksta trešajā atsevišķā reģistrā "out", kas parasti ir ar $m = 2n$ kubitiem, un bāzes stāvoklī $|0\rangle^{\otimes m}$ 
Tiek realizēts:
$|a\rangle|b\rangle|0\rangle \mapsto |a\rangle|b\rangle|a\cdot b\rangle.$

Darbības soļi:
1) uz "out" uzliek $\mathrm{QFT}$  
2) ar kontrolētām fāžu rotācijām uz "out" pārveido produktu fāzēs, izmantojot identitāti $a\cdot b=\sum_{i,j} a_i b_j 2^{i+j}$,  
3) uz "out" uzliek $\mathrm{QFT}^\dagger$, lai fāzes pārvērstu atpakaļ par parasto bināro rezultātu.

## Piemērs ar 3 bitiem katrā reģistrā: 
$a=5 = 101_2$ un $b=3 = 011_2$.

- $a_2=1,\ a_1=0,\ a_0=1$
- $b_2=0,\ b_1=1,\ b_0=1$

Identitāte:
$a\cdot b=\sum_{i=0}^{2}\sum_{j=0}^{2} a_i b_j 2^{i+j}.$

Nenulles rezultāts veidojas kur $a_i=b_j=1$:
- $(0,0)\Rightarrow 2^{0+0}=1$
- $(0,1)\Rightarrow 2^{0+1}=2$
- $(2,0)\Rightarrow 2^{2+0}=4$
- $(2,1)\Rightarrow 2^{2+1}=8$

$a\cdot b = 1+2+4+8 = 15$,
jeb: $15 = 001111_2$.

### "out" reģistrs
"out" sākumā ir $|000000\rangle$ .

$\mathrm{QFT}|0\rangle=\frac{1}{\sqrt{2^6}}\sum_{k=0}^{63}|k\rangle.$

Fāžu solis realizē transformāciju:
$|k\rangle \mapsto e^{2\pi i (a\cdot b)k/2^6}|k\rangle.$

Šajā piemērā $a\cdot b=15$, tāpēc:
$\frac{1}{\sqrt{64}}\sum_{k=0}^{63}|k\rangle
\ \mapsto\
\frac{1}{\sqrt{64}}\sum_{k=0}^{63}e^{2\pi i \cdot 15k/64}|k\rangle
=\mathrm{QFT}|15\rangle.$

Beigās inverse QFT atgriež parasto bināro stāvokli:
$\mathrm{QFT}^\dagger(\mathrm{QFT}|15\rangle)=|15\rangle.$

Tātad "out" kļūst par $|001111\rangle$, bet "a"  un "b" paliek nemainīgi.


In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RGQFTMultiplier

sim = AerSimulator()
mult = RGQFTMultiplier(num_state_qubits=3)

# -------------------------------------------------
# Piemērs 1
a1 = QuantumRegister(3, "a")
b1 = QuantumRegister(3, "b")
out1 = QuantumRegister(6, "out") # out ir 6 kubiti (2*n)
c_out1 = ClassicalRegister(6, "c_out")
qc1 = QuantumCircuit(a1, b1, out1, c_out1)

# a = 5 = 101
qc1.x(a1[0]) 
qc1.x(a1[2])
# b = 3 = 011
qc1.x(b1[0]) 
qc1.x(b1[1])

qc1.append(mult, [*a1, *b1, *out1])
qc1.measure(out1, c_out1)

tqc1 = transpile(qc1, sim)
counts1 = sim.run(tqc1, shots=1024).result().get_counts()
print("RGQFTMultiplier: a=5, b=3 -> out=15")
print("Rezultāts: ", counts1)
print(qc1.draw("text"))


# -------------------------------------------------
# Piemērs 2
a2 = QuantumRegister(3, "a")
b2 = QuantumRegister(3, "b")
out2 = QuantumRegister(6, "out")
c_out2 = ClassicalRegister(6, "c_out")
qc2 = QuantumCircuit(a2, b2, out2, c_out2)

# a = (|000> + |100>)/sqrt(2)
qc2.h(a2[2])

# b = 3 = 011
qc2.x(b2[0]); qc2.x(b2[1])

qc2.append(mult, [*a2, *b2, *out2])
qc2.measure(out2, c_out2)

tqc2= transpile(qc2, sim)
counts2 = sim.run(tqc2, shots=1024).result().get_counts()
print("RGQFTMultiplier: a = 0 vai 4, b=3 -> out= 0 vai 12")
print("Rezultāts: ", counts2)
print(qc2.decompose().decompose().draw("text"))

RGQFTMultiplier: a=5, b=3 -> out=15
Rezultāts:  {'001111': 1024}
         ┌───┐┌───────────────────┐                  
    a_0: ┤ X ├┤0                  ├──────────────────
         └───┘│                   │                  
    a_1: ─────┤1                  ├──────────────────
         ┌───┐│                   │                  
    a_2: ┤ X ├┤2                  ├──────────────────
         ├───┤│                   │                  
    b_0: ┤ X ├┤3                  ├──────────────────
         ├───┤│                   │                  
    b_1: ┤ X ├┤4                  ├──────────────────
         └───┘│                   │                  
    b_2: ─────┤5                  ├──────────────────
              │   RGQFTMultiplier │┌─┐               
  out_0: ─────┤6                  ├┤M├───────────────
              │                   │└╥┘┌─┐            
  out_1: ─────┤7                  ├─╫─┤M├────────────
              │                   │ ║ └╥┘┌─┐         
  out_2: ─────┤8 